<a href="https://colab.research.google.com/github/akgupta63/CODSOFT_TASK3/blob/main/CUSTOMER_CHURN_PREDICTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Prediction
This notebook builds machine learning models to predict customer churn using historical customer data.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import os

# Define dataset path
file_path = '/content/Churn_Modelling.csv'
if not os.path.exists(file_path):
    # fallback to current working directory
    file_path = 'Churn_Modelling.csv'

# Load the dataset
df = pd.read_csv(file_path)
display(df.head())

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Inspect data types and missing values
print(df.info())
print("\nMissing values count per column:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB
None

Missing values count per column:
RowNumber          0
CustomerId         0
Surname      

In [4]:
# Drop irrelevant columns for modeling (e.g., RowNumber, CustomerId, Surname)
columns_to_drop = ['RowNumber', 'CustomerId', 'Surname']
existing_drops = [col for col in columns_to_drop if col in df.columns]
df_cleaned = df.drop(columns=existing_drops)

# Define target and features
target_col = 'Exited'
X = df_cleaned.drop(columns=[target_col])
y = df_cleaned[target_col]

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical features: {num_cols}")
print(f"Categorical features: {cat_cols}")

Numerical features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical features: ['Geography', 'Gender']


In [5]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Train set shape: (8000, 10)
Test set shape: (2000, 10)


In [7]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Train and evaluate models
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Evaluate
    print(f"==================== {name} ====================")
    print(classification_report(y_test, y_pred))
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n")

==================== Logistic Regression ====================
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.59      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.71      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000

ROC AUC Score: 0.7748

==================== Random Forest ====================
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.78      0.46      0.58       407

    accuracy                           0.86      2000
   macro avg       0.83      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000

ROC AUC Score: 0.8522

==================== Gradient Boosting ====================
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1 

# Customer Churn Prediction Project

This project focuses on predicting customer churn using historical customer data from a subscription-based business model. It evaluates and compares the performance of multiple machine learning classifiers to determine the best model for detecting customers at risk of leaving.

## Table of Contents
1. [Project Overview](#project-overview)
2. [Dataset Description](#dataset-description)
3. [Data Preprocessing](#data-preprocessing)
4. [Model Architecture](#model-architecture)
5. [Model Evaluation Results](#model-evaluation-results)
6. [How to Run](#how-to-run)

---

## Project Overview
Customer churn represents a major revenue leakage for subscription businesses. Detecting potential churners beforehand allows companies to proactively target them with retention campaigns. In this project, we implement and benchmark three popular algorithms:
- **Logistic Regression** (Linear baseline)
- **Random Forest** (Ensemble bagging)
- **Gradient Boosting** (Ensemble boosting)

## Dataset Description
The dataset used is `Churn_Modelling.csv`. It contains 10,000 customer records with the following characteristics:
- **Demographics**: Geography, Gender, Age.
- **Account Metrics**: Credit Score, Tenure, Account Balance, Number of Products, Estimated Salary, Active Membership Status, and Credit Card Ownership.
- **Target Variable**: `Exited` (1 if the customer churned, 0 otherwise).

## Data Preprocessing
To prepare the dataset for machine learning, the following pipeline was built:
1. **Feature Dropping**: Irrelevant identifier columns (`RowNumber`, `CustomerId`, `Surname`) were removed.
2. **Feature Types Identification**: Separated into Numerical and Categorical feature lists.
3. **Feature Scaling**: Numerical attributes were standardized using `StandardScaler`.
4. **Categorical Encoding**: Categorical variables (`Geography`, `Gender`) were converted into numerical values using `OneHotEncoder`.
5. **Data Splitting**: Split into 80% Training and 20% Testing sets using stratification to handle target imbalance.

## Model Architecture
We leveraged scikit-learn's `Pipeline` API to bundle preprocessing and estimator training seamlessly, ensuring no data leakage occurred between training and validation steps.

## Model Evaluation Results

| Model | Accuracy | Recall (Class 1) | F1-Score (Class 1) | ROC AUC |
| :--- | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 81% | 19% | 0.28 | 0.7748 |
| **Random Forest** | 86% | 46% | 0.58 | 0.8522 |
| **Gradient Boosting** | **87%** | **49%** | **0.60** | **0.8708** |

### Conclusion
**Gradient Boosting Classifier** is the champion model with the highest ROC AUC of **0.8708** and the best balance of Precision and Recall for detecting churned customers.